In [35]:
# Import packages
import os
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold

In [36]:
# Load in dataset
path = Path("../../../../local3/sswee/music_download/physionet.org/files/music-sudden-cardiac-death/1.0.1/subject-info.csv")

df = pd.read_csv(
    path,
    sep=";",          # correct delimiter
    decimal=",",      # European decimal format
    engine="python",  # handle irregular formatting
    na_values=["", "NA"] # handles missing values
)

# Clean (tabs inside numbers)
df = df.replace(r"\t", ".", regex=True)

# Convert age to numeric
df["Age"] = (
    df["Age"]
    .astype(str)
    .str.strip()
    .str.replace(",", ".", regex=False)
)
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")

print(df.shape) # 992 patients with 103 columns

df.head()

(992, 103)


,Patient ID,Follow-up period from enrollment (days),days_4years,Exit of the study,Cause of death,Age,Gender (male=1),Weight (kg),Height (cm),Body Mass Index (Kg/m2),...,Angiotensin-II receptor blocker (yes=1),Anticoagulants/antitrombotics (yes=1),Betablockers (yes=1),Digoxin (yes=1),Loop diuretics (yes=1),Spironolactone (yes=1),Statins (yes=1),Hidralazina (yes=1),ACE inhibitor (yes=1),Nitrovasodilator (yes=1)
0,P0001,2065,1460,NaN,0,58.0,1,83,163,31.2,...,0,1,1,1,1,0,0,0,1,0
1,P0002,2045,1460,NaN,0,58.0,1,74,160,28.9,...,1,1,1,0,0,0,1,0,0,0
2,P0003,2044,1460,NaN,0,69.0,1,83,174,27.4,...,1,1,1,1,1,0,0,0,0,0
3,P0004,2044,1460,NaN,0,56.0,0,84,165,30.9,...,1,1,1,0,1,1,0,0,0,0
4,P0005,2043,1460,NaN,0,70.0,1,97,183,29.0,...,0,1,1,0,1,0,1,0,1,1


In [37]:
df["Patient ID"] = df["Patient ID"].str.replace("P", "", regex=False)
df["Patient ID"] = df["Patient ID"].str.strip()

df["Patient ID"] = (
    df["Patient ID"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)  # remove trailing .0
    .str.zfill(4)
)

In [38]:
# Patient selection
mask = (
    (df["Holter available"] != 0) &  # Select patients with Holter
    (df["Prior implantable device"] == 0) &  # Remove pacemaker patients
    ((df["Exit of the study"].ne(2)) | (df["Exit of the study"].isna()))  # Remove cardiac transplant
)

df2 = df[mask].copy()

# Exit of study is 0 for survivor
df2["Exit of the study"] = df2["Exit of the study"].fillna(0).astype(int)

# Combine Cause of death codes:
# 6 and 7 -> Pump failure death
df2.loc[df2["Cause of death"].isin([6, 7]), "Cause of death"] = 6

# Time to event
df2["time_to_event_days"] = df2[
    ["Follow-up period from enrollment (days)", "days_4years"]
].min(axis=1)

# Any cardiac death (SCD or PFD)
df2["event_cardiac"] = df2["Cause of death"].isin([3, 6]).astype(int)

# Cause-specific events
df2["event_cardiac_SCD"] = (df2["Cause of death"] == 3).astype(int)
df2["event_cardiac_PFD"] = (df2["Cause of death"] == 6).astype(int)

print("Cause of death counts:")
print(df2["Cause of death"].value_counts().sort_index())

print("\nEvent counts:")
print(df2[["event_cardiac", "event_cardiac_SCD", "event_cardiac_PFD"]].sum())

# Table of number of patients per outcome
cause_counts = df2["Cause of death"].value_counts().sort_index()
print(cause_counts)

# Note: Original paper reports 94/996 SCDs (9.4%) and 111/996 PFDs (11.1%)
# Final counts align very well with 74/796 SCDs (9.3%) and 87/796 PFDs (10.9%)

Cause of death counts:
Cause of death
0    585
1     50
3     74
6     87
Name: count, dtype: int64

Event counts:
event_cardiac        161
event_cardiac_SCD     74
event_cardiac_PFD     87
dtype: int64
Cause of death
0    585
1     50
3     74
6     87
Name: count, dtype: int64


In [39]:
df2.head()

,Patient ID,Follow-up period from enrollment (days),days_4years,Exit of the study,Cause of death,Age,Gender (male=1),Weight (kg),Height (cm),Body Mass Index (Kg/m2),...,Loop diuretics (yes=1),Spironolactone (yes=1),Statins (yes=1),Hidralazina (yes=1),ACE inhibitor (yes=1),Nitrovasodilator (yes=1),time_to_event_days,event_cardiac,event_cardiac_SCD,event_cardiac_PFD
0,0001,2065,1460,0,0,58.0,1,83,163,31.2,...,1,0,0,0,1,0,1460,0,0,0
1,0002,2045,1460,0,0,58.0,1,74,160,28.9,...,0,0,1,0,0,0,1460,0,0,0
2,0003,2044,1460,0,0,69.0,1,83,174,27.4,...,1,0,0,0,0,0,1460,0,0,0
3,0004,2044,1460,0,0,56.0,0,84,165,30.9,...,1,1,0,0,0,0,1460,0,0,0
5,0006,2043,1460,0,0,70.0,1,83,165,30.5,...,1,1,0,0,1,1,1460,0,0,0


In [40]:
# --------------------------------------------------
# Create stratification label
# 0 = censored
# 1 = SCD
# 2 = PFD
# --------------------------------------------------
def map_outcome(cause):
    if cause == 3:
        return 1
    elif cause == 6:
        return 2
    else:
        return 0

df2["outcome_strata"] = df2["Cause of death"].apply(map_outcome)

# --------------------------------------------------
# Stratified K-Fold
# --------------------------------------------------
N_FOLDS = 5
skf = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=42
)

df2 = df2.reset_index(drop=True)

df2["fold"] = -1

for fold, (_, val_idx) in enumerate(
    skf.split(df2, df2["outcome_strata"])
):
    df2.loc[val_idx, "fold"] = fold

In [41]:
# -----------------------------
# Keep only required columns
# -----------------------------
keep_cols = [
    "Patient ID",
    "fold",
    "time_to_event_days",
    "Cause of death",
]

df_clean = df2[keep_cols].copy()

# -----------------------------
# Sanity checks
# -----------------------------

# Ensure Patient ID is string (important for folder matching)
df_clean["Patient ID"] = df_clean["Patient ID"].astype(str).str.zfill(4)

# Ensure fold is int
df_clean["fold"] = df_clean["fold"].astype(int)

# Ensure time is float
df_clean["time_to_event_days"] = df_clean["time_to_event_days"].astype(float)

# Ensure Cause of death is int
df_clean["Cause of death"] = df_clean["Cause of death"].astype(int)

# -----------------------------
# Optional: filter to supported causes only
# -----------------------------
# Keep only survivors, SCD, and PFD
df_clean = df_clean[df_clean["Cause of death"].isin([0, 3, 6])]

# -----------------------------
# Final sanity logging
# -----------------------------
print("Final cohort size:", len(df_clean))
print(df_clean["Cause of death"].value_counts())

Final cohort size: 746
Cause of death
0    585
6     87
3     74
Name: count, dtype: int64


In [42]:
print(
    df_clean.groupby(["fold", "Cause of death"])
            .size()
            .unstack(fill_value=0)
)


Cause of death    0   3   6
fold                       
0               118  15  18
1               120  15  17
2               118  15  17
3               114  15  17
4               115  14  18


In [43]:
# Save labels
path = Path("../../../../local3/sswee/music_download/physionet.org/files/music-sudden-cardiac-death/1.0.1")

df_clean.to_csv(path / "music_survival_5cv_cause_specific.csv", index = False)

print(f"Saved final training CSV")
print(df_final.head())

Saved final training CSV


NameError: name 'df_final' is not defined